<a href="https://colab.research.google.com/github/SahilRathi-AI/Gender-Bias-Salary-Ethics-Audit/blob/main/Sahil_Rathi_GH1031080_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.metrics import confusion_matrix, classification_report

In [3]:
import glob
csv_files = glob.glob(
    "/content/drive/MyDrive/CICIDS2017/*.csv"
)
print("Files found:", len(csv_files))
frames = []
for file in csv_files:
    print("Loading:", os.path.basename(file))
    temp = pd.read_csv(
        file,
        low_memory=False
    )
    frames.append(temp)
df = pd.concat(
    frames,
    ignore_index=True
)
print("Combined Shape:", df.shape)

df.head()

Files found: 8
Loading: Wednesday-workingHours.pcap_ISCX.csv
Loading: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Loading: Tuesday-WorkingHours.pcap_ISCX.csv
Loading: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Loading: Monday-WorkingHours.pcap_ISCX.csv
Loading: Friday-WorkingHours-Morning.pcap_ISCX.csv
Loading: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Loading: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Combined Shape: (2830743, 79)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,80,38308,1,1,6,6,6,6,6.000000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,389,479,11,5,172,326,79,0,15.636364,31.449238,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,88,1095,10,6,3150,3150,1575,0,315.000000,632.561635,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,389,15206,17,12,3452,6660,1313,0,203.058823,425.778474,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,88,1092,9,6,3150,3152,1575,0,350.000000,694.509719,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [4]:
df.columns = df.columns.str.strip()
print("Columns:", len(df.columns))
print(df.columns.tolist())

Columns: 79
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'EC

In [5]:
print(df["Label"].value_counts())

Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [6]:
df["Label"] = df["Label"].apply(
    lambda x: 0 if x == "BENIGN" else 1
)
print(df["Label"].value_counts())

Label
0    2273097
1     557646
Name: count, dtype: int64


In [7]:
columns_to_remove = [
    "Label",
    "Flow ID",
    "Source IP",
    "Destination IP",
    "Timestamp"
]
existing_columns = [
    col for col in columns_to_remove
    if col in df.columns
]
X = df.drop(
    columns=existing_columns
)
print(X.shape)

(2830743, 78)


In [8]:
benign_train = df[df["Label"] == 0].copy()
print("Benign Shape:", benign_train.shape)

Benign Shape: (2273097, 79)


In [9]:
X_train = benign_train.drop(
    columns=["Label"]
)
X_train = X_train.select_dtypes(
    include=[np.number]
)
print(X_train.shape)

(2273097, 78)


In [10]:
X_train.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)
X_train.fillna(
    X_train.median(),
    inplace=True
)
print(X_train.shape)

(2273097, 78)


In [11]:
print(df["Label"].value_counts())
print(X_train.shape)

Label
0    2273097
1     557646
Name: count, dtype: int64
(2273097, 78)


In [12]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
print("Scaling completed")

Scaling completed


# ============================================================
# Save training features and scaler
# ============================================================


In [13]:
import os
import joblib
os.makedirs(
    "/content/drive/MyDrive/CICIDS2017/models",
    exist_ok=True
)
joblib.dump(
    list(X_train.columns),
    "/content/drive/MyDrive/CICIDS2017/models/training_features.joblib"
)
joblib.dump(
    scaler,
    "/content/drive/MyDrive/CICIDS2017/models/scaler.joblib"
)
print("Saved successfully")

Saved successfully


In [14]:
import os
print(os.listdir("/content/drive/MyDrive/CICIDS2017/models"))

['training_features.joblib', 'scaler.joblib']


In [15]:
test_data = df.copy()
X_test = test_data[X_train.columns].copy()
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)
X_test.fillna(X_train.median(), inplace=True)
y_true = test_data["Label"]
X_test_scaled = scaler.transform(X_test)
print("Testing shape:", X_test_scaled.shape)

Testing shape: (2830743, 78)


In [16]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
iso_model = IsolationForest(
    n_estimators=100,
    contamination=0.20,
    random_state=42,
    n_jobs=-1
)
iso_model.fit(X_train_scaled)
iso_raw = iso_model.predict(X_test_scaled)
iso_pred = np.where(iso_raw == 1, 0, 1)
print("Isolation Forest Results")
print(confusion_matrix(y_true, iso_pred))
print(classification_report(y_true, iso_pred, target_names=["Benign", "Attack"], zero_division=0))

Isolation Forest Results
[[1818477  454620]
 [ 299610  258036]]
              precision    recall  f1-score   support

      Benign       0.86      0.80      0.83   2273097
      Attack       0.36      0.46      0.41    557646

    accuracy                           0.73   2830743
   macro avg       0.61      0.63      0.62   2830743
weighted avg       0.76      0.73      0.75   2830743



In [17]:
from sklearn.linear_model import SGDOneClassSVM
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

In [18]:
sgd_ocsvm = SGDOneClassSVM(
    nu=0.05,
    random_state=42
)
sgd_ocsvm.fit(X_train_scaled)
svm_raw_pred = sgd_ocsvm.predict(X_test_scaled)
svm_pred = np.where(svm_raw_pred == 1, 0, 1)
print("SGD One-Class SVM Results")
print(confusion_matrix(y_true, svm_pred))
print(classification_report(
    y_true,
    svm_pred,
    target_names=["Benign", "Attack"],
    zero_division=0
))

SGD One-Class SVM Results
[[2119360  153737]
 [ 510348   47298]]
              precision    recall  f1-score   support

      Benign       0.81      0.93      0.86   2273097
      Attack       0.24      0.08      0.12    557646

    accuracy                           0.77   2830743
   macro avg       0.52      0.51      0.49   2830743
weighted avg       0.69      0.77      0.72   2830743



In [19]:
!pip install tensorflow

In [20]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model
input_dim = X_train_scaled.shape[1]
input_layer = Input(shape=(input_dim,))
x = Dense(64, activation="relu")(input_layer)
x = Dense(32, activation="relu")(x)
x = Dense(16, activation="relu")(x)
x = Dense(32, activation="relu")(x)
x = Dense(64, activation="relu")(x)
output_layer = Dense(input_dim, activation="linear")(x)
autoencoder = Model(input_layer, output_layer)
autoencoder.compile(
    optimizer="adam",
    loss="mse"
)
autoencoder.fit(
    X_train_scaled,
    X_train_scaled,
    epochs=15,
    batch_size=512,
    validation_split=0.1,
    verbose=1
)

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


Epoch 1/15
3996/3996 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - loss: 0.1843 - val_loss: 0.2553
Epoch 2/15
3996/3996 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 0.1099 - val_loss: 0.2033
Epoch 3/15
3996/3996 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - loss: 0.0838 - val_loss: 0.1683
Epoch 4/15
3996/3996 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - loss: 0.0724 - val_loss: 0.1507
Epoch 5/15
3996/3996 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - loss: 0.0598 - val_loss: 0.1391
Epoch 6/15
3996/3996 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - loss: 0.0599 - val_loss: 0.1401
Epoch 7/15
3996/3996 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - loss: 0.0491 - val_loss: 0.1291
Epoch 8/15
3996/3996 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - loss: 0.0541 - val_loss: 0.1237
Epoch 9/15
3996/3996 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - loss: 0.0401 - val_loss: 0.0972
Epoch 10/15
3996/3996 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - loss: 0.0480 - val_loss: 0.1045
Epoch 11/15
3996/3996 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - loss: 0.0355 - val_loss: 0.1924
Epoch 12/15
3996/3996 ━━━━━━━━

In [21]:
reconstruction = autoencoder.predict(X_test_scaled, batch_size=512)
mse = np.mean(np.square(X_test_scaled - reconstruction), axis=1)
threshold = np.percentile(mse, 80)
ae_pred = np.where(mse > threshold, 1, 0)
print(confusion_matrix(y_true, ae_pred))
print(classification_report(y_true, ae_pred, target_names=["Benign", "Attack"], zero_division=0))

5529/5529 ━━━━━━━━━━━━━━━━━━━━ 4s 643us/step
[[1998276  274821]
 [ 266318  291328]]
              precision    recall  f1-score   support

      Benign       0.88      0.88      0.88   2273097
      Attack       0.51      0.52      0.52    557646

    accuracy                           0.81   2830743
   macro avg       0.70      0.70      0.70   2830743
weighted avg       0.81      0.81      0.81   2830743



In [22]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import os

SAVE_DIR = "/content/drive/MyDrive/Result/2017"
os.makedirs(SAVE_DIR, exist_ok=True)

results_2017 = []

def safe_add_2017(model_name, y_true_data, y_pred_data):
    if len(y_true_data) == len(y_pred_data):
        results_2017.append({
            "Dataset": "CICIDS2017",
            "Model": model_name,
            "Accuracy": accuracy_score(y_true_data, y_pred_data),
            "Attack Precision": precision_score(y_true_data, y_pred_data, zero_division=0),
            "Attack Recall": recall_score(y_true_data, y_pred_data, zero_division=0),
            "Attack F1": f1_score(y_true_data, y_pred_data, zero_division=0)
        })
        print(model_name, "added")
    else:
        print(model_name, "skipped because lengths do not match")

safe_add_2017("Isolation Forest", y_true, iso_pred)
safe_add_2017("SGD One-Class SVM", y_true, svm_pred)
safe_add_2017("Autoencoder", y_true, ae_pred)

# These may belong to 2018, so only add if length matches
if "vae_pred_2017" in globals():
    safe_add_2017("VAE", y_true, vae_pred_2017)

if "svdd_pred_2017" in globals():
    safe_add_2017("Deep SVDD-style", y_true, svdd_pred_2017)

results_2017 = pd.DataFrame(results_2017)
display(results_2017)

results_2017.to_csv(
    f"{SAVE_DIR}/results_2017.csv",
    index=False
)

print("2017 results saved")

Isolation Forest added
SGD One-Class SVM added
Autoencoder added


,Dataset,Model,Accuracy,Attack Precision,Attack Recall,Attack F1
0,CICIDS2017,Isolation Forest,0.733558,0.362077,0.462724,0.406259
1,CICIDS2017,SGD One-Class SVM,0.765403,0.235272,0.084817,0.124685
2,CICIDS2017,Autoencoder,0.808835,0.514578,0.522425,0.518472


2017 results saved


# 2018


In [23]:
cicids2018_files = glob.glob("/content/drive/MyDrive/CICIDS_2018/*.csv")
print("CICIDS2018 files found:", len(cicids2018_files))
for file in cicids2018_files:
    print(os.path.basename(file))

CICIDS2018 files found: 10
Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
Friday-23-02-2018_TrafficForML_CICFlowMeter.csv
Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv
Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv
Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv
Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv
Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv
Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv


In [24]:
for file in cicids2018_files:
    temp = pd.read_csv(file, nrows=5, low_memory=False)
    temp.columns = temp.columns.str.strip()
    print("\nFile:", os.path.basename(file))
    print("Columns:", len(temp.columns))
    print("Label column:", [c for c in temp.columns if "label" in c.lower()])


File: Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
Columns: 80
Label column: ['Label']

File: Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
Columns: 80
Label column: ['Label']

File: Friday-23-02-2018_TrafficForML_CICFlowMeter.csv
Columns: 80
Label column: ['Label']

File: Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
Columns: 84
Label column: ['Label']

File: Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv
Columns: 80
Label column: ['Label']

File: Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv
Columns: 80
Label column: ['Label']

File: Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv
Columns: 80
Label column: ['Label']

File: Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv
Columns: 80
Label column: ['Label']

File: Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv
Columns: 80
Label column: ['Label']

File: Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv
Columns: 80
Label column: ['Label']


In [25]:
sample_2018 = pd.read_csv(cicids2018_files[0], nrows=1000, low_memory=False)
sample_2018.columns = sample_2018.columns.str.strip()
cicids2017_features = set(X_train.columns)
cicids2018_features = set(sample_2018.columns)
common_features = sorted(
    list(cicids2017_features.intersection(cicids2018_features))
)
print("CICIDS2017 features:", len(cicids2017_features))
print("CICIDS2018 features:", len(cicids2018_features))
print("Common features:", len(common_features))
print(common_features)

CICIDS2017 features: 78
CICIDS2018 features: 80
Common features: 27
['Active Max', 'Active Mean', 'Active Min', 'Active Std', 'Bwd IAT Max', 'Bwd IAT Mean', 'Bwd IAT Min', 'Bwd IAT Std', 'Bwd PSH Flags', 'Bwd URG Flags', 'CWE Flag Count', 'Down/Up Ratio', 'Flow Duration', 'Flow IAT Max', 'Flow IAT Mean', 'Flow IAT Min', 'Flow IAT Std', 'Fwd IAT Max', 'Fwd IAT Mean', 'Fwd IAT Min', 'Fwd IAT Std', 'Fwd PSH Flags', 'Fwd URG Flags', 'Idle Max', 'Idle Mean', 'Idle Min', 'Idle Std']


In [26]:
print("Common features:", len(common_features))
print(common_features)

Common features: 27
['Active Max', 'Active Mean', 'Active Min', 'Active Std', 'Bwd IAT Max', 'Bwd IAT Mean', 'Bwd IAT Min', 'Bwd IAT Std', 'Bwd PSH Flags', 'Bwd URG Flags', 'CWE Flag Count', 'Down/Up Ratio', 'Flow Duration', 'Flow IAT Max', 'Flow IAT Mean', 'Flow IAT Min', 'Flow IAT Std', 'Fwd IAT Max', 'Fwd IAT Mean', 'Fwd IAT Min', 'Fwd IAT Std', 'Fwd PSH Flags', 'Fwd URG Flags', 'Idle Max', 'Idle Mean', 'Idle Min', 'Idle Std']


In [27]:
for file in cicids2018_files:
    temp = pd.read_csv(file, nrows=5, low_memory=False)
    temp.columns = temp.columns.str.strip()
    print("\nFile:", os.path.basename(file))
    print("Shape sample:", temp.shape)
    print("Columns:", len(temp.columns))
    print("Label column:", [c for c in temp.columns if "label" in c.lower()])


File: Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
Shape sample: (5, 80)
Columns: 80
Label column: ['Label']

File: Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
Shape sample: (5, 80)
Columns: 80
Label column: ['Label']

File: Friday-23-02-2018_TrafficForML_CICFlowMeter.csv
Shape sample: (5, 80)
Columns: 80
Label column: ['Label']

File: Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
Shape sample: (5, 84)
Columns: 84
Label column: ['Label']

File: Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv
Shape sample: (5, 80)
Columns: 80
Label column: ['Label']

File: Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv
Shape sample: (5, 80)
Columns: 80
Label column: ['Label']

File: Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv
Shape sample: (5, 80)
Columns: 80
Label column: ['Label']

File: Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv
Shape sample: (5, 80)
Columns: 80
Label column: ['Label']

File: Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv
Shape sample: (5, 80)
Colum

In [ ]:
all_2018_frames = []
for file in cicids2018_files:
    print("Loading:", os.path.basename(file))
    temp = pd.read_csv(
        file,
        low_memory=False
    )
    temp.columns = temp.columns.str.strip()
    all_2018_frames.append(temp)

cicids2018_data = pd.concat(
    all_2018_frames,
    ignore_index=True
)
print("\nAll files combined")
print("Shape:", cicids2018_data.shape)
cicids2018_data.head()

del all_2018_frames
import gc
gc.collect()
print("\nAll files combined")
print("Shape:", cicids2018_data.shape)

cicids2018_data.head()

Loading: Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
Loading: Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
Loading: Friday-23-02-2018_TrafficForML_CICFlowMeter.csv
Loading: Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv


In [ ]:
cicids2018_data.info()

In [ ]:
print(cicids2018_data["Label"].value_counts())

In [ ]:
missing_values = cicids2018_data.isnull().sum()
print(missing_values[missing_values > 0])

In [ ]:
numeric_2018 = cicids2018_data.select_dtypes(include=[np.number])
inf_values = np.isinf(numeric_2018).sum()
print(inf_values[inf_values > 0])

In [ ]:
cicids2017_features = set(X_train.columns)
cicids2018_features = set(cicids2018_data.columns)
common_features = sorted(
    list(cicids2017_features.intersection(cicids2018_features))
)
print("CICIDS2017 features:", len(cicids2017_features))
print("CICIDS2018 features:", len(cicids2018_features))
print("Common features:", len(common_features))
print(common_features)

In [ ]:
X_2018 = cicids2018_data[common_features].copy()
X_2018.replace([np.inf, -np.inf], np.nan, inplace=True)
X_2018.fillna(
    X_train[common_features].median(),
    inplace=True
)
print("Aligned 2018 shape:", X_2018.shape)
print("Missing after cleaning:", X_2018.isnull().sum().sum())

In [ ]:
y_2018 = cicids2018_data["Label"].apply(
    lambda x: 0 if str(x).strip().upper() in ["BENIGN", "NORMAL"] else 1
)
print(y_2018.value_counts())

In [ ]:
X_train_common = X_train[common_features].copy()
X_train_common.replace([np.inf, -np.inf], np.nan, inplace=True)
X_train_common.fillna(X_train_common.median(), inplace=True)
print("Common training shape:", X_train_common.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler
common_scaler = StandardScaler()
X_train_common_scaled = common_scaler.fit_transform(X_train_common)
print("Common scaler fitted")

In [ ]:
X_2018 = cicids2018_data[common_features].copy()
for col in X_2018.columns:
    X_2018[col] = pd.to_numeric(X_2018[col], errors="coerce")
X_2018.replace([np.inf, -np.inf], np.nan, inplace=True)
X_2018.fillna(
    X_train_common.median(),
    inplace=True
)
print("Missing values:", X_2018.isnull().sum().sum())
X_2018_scaled = common_scaler.transform(X_2018)
print("2018 scaled shape:", X_2018_scaled.shape)

In [ ]:
print(X_2018.select_dtypes(include=["object"]).columns)

In [ ]:
X_2018 = cicids2018_data[common_features].copy()
for col in X_2018.columns:
    X_2018[col] = pd.to_numeric(
        X_2018[col],
        errors="coerce"
    )
X_2018.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)
X_2018.fillna(
    X_train_common.median(),
    inplace=True
)
X_2018_scaled = common_scaler.transform(X_2018)

print("2018 scaled shape:", X_2018_scaled.shape)

In [ ]:
X_2018 = cicids2018_data[common_features].copy()
for col in X_2018.columns:
    X_2018[col] = pd.to_numeric(
        X_2018[col],
        errors="coerce"
    )
X_2018.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)
X_2018.fillna(
    X_train_common.median(),
    inplace=True
)
X_2018_scaled = common_scaler.transform(X_2018)
print("2018 scaled shape:", X_2018_scaled.shape)

In [ ]:
iso_common = IsolationForest(
    n_estimators=100,
    contamination=0.20,
    random_state=42,
    n_jobs=-1
)
iso_common.fit(X_train_common_scaled)

In [ ]:
pred_2018 = iso_common.predict(X_2018_scaled)
y_pred_2018 = np.where(pred_2018 == 1, 0, 1)
print(confusion_matrix(y_2018, y_pred_2018))
print(classification_report(y_2018, y_pred_2018, target_names=["Benign", "Attack"], zero_division=0))

In [ ]:
cross_results = []
cross_results.append({
    "Model": "Isolation Forest",
    "Dataset": os.path.basename(cicids2018_files[0]),
    "Accuracy": 0.56,
    "Attack Recall": 0.00,
    "Attack F1": 0.00
})

In [ ]:
input_dim = X_train_common_scaled.shape[1]
input_layer = Input(shape=(input_dim,))
x = Dense(64, activation="relu")(input_layer)
x = Dense(32, activation="relu")(x)
x = Dense(16, activation="relu")(x)
x = Dense(32, activation="relu")(x)
x = Dense(64, activation="relu")(x)
output_layer = Dense(input_dim, activation="linear")(x)
ae_common = Model(input_layer, output_layer)
ae_common.compile(optimizer="adam", loss="mse")
ae_common.fit(
    X_train_common_scaled,
    X_train_common_scaled,
    epochs=10,
    batch_size=512,
    validation_split=0.1,
    verbose=1
)

In [ ]:
train_recon = ae_common.predict(X_train_common_scaled, batch_size=512)
train_mse = np.mean(np.square(X_train_common_scaled - train_recon), axis=1)
threshold = np.percentile(train_mse, 80)
test_recon = ae_common.predict(X_2018_scaled, batch_size=512)
test_mse = np.mean(np.square(X_2018_scaled - test_recon), axis=1)
ae_2018_pred = np.where(test_mse > threshold, 1, 0)

print(confusion_matrix(y_2018, ae_2018_pred))
print(classification_report(y_2018, ae_2018_pred, target_names=["Benign", "Attack"], zero_division=0))

In [ ]:
from sklearn.linear_model import SGDOneClassSVM
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
sgd_common = SGDOneClassSVM(
    nu=0.05,
    random_state=42
)
# train only on CICIDS2017 benign common features
sgd_common.fit(X_train_common_scaled)

# test on CICIDS2018
svm_2018_raw = sgd_common.predict(X_2018_scaled)
svm_2018_pred = np.where(svm_2018_raw == 1, 0, 1)

print("SGD One-Class SVM: CICIDS2017 → CSE-CIC-IDS2018")
print(confusion_matrix(y_2018, svm_2018_pred))
print(classification_report(
    y_2018,
    svm_2018_pred,
    target_names=["Benign", "Attack"],
    zero_division=0
))

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
sgd_2018_result = {
    "Model": "SGD One-Class SVM",
    "Dataset": "CICIDS2017 → CSE-CIC-IDS2018",
    "Accuracy": accuracy_score(y_2018, svm_2018_pred),
    "Attack Precision": precision_score(y_2018, svm_2018_pred, zero_division=0),
    "Attack Recall": recall_score(y_2018, svm_2018_pred, zero_division=0),
    "Attack F1": f1_score(y_2018, svm_2018_pred, zero_division=0)
}
sgd_2018_result

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Lambda
from tensorflow.keras.models import Model
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

input_dim = X_train_common_scaled.shape[1]
latent_dim = 8
vae_input = Input(shape=(input_dim,))

x = Dense(64, activation="relu")(vae_input)
x = Dense(32, activation="relu")(x)
z_mean = Dense(latent_dim)(x)
z_log_var = Dense(latent_dim)(x)

def sample_z(args):
    z_mean, z_log_var = args
    epsilon = tf.random.normal(shape=tf.shape(z_mean))
    return z_mean + tf.exp(0.5 * z_log_var) * epsilon

z = Lambda(sample_z)([z_mean, z_log_var])

x = Dense(32, activation="relu")(z)
x = Dense(64, activation="relu")(x)
vae_output = Dense(input_dim, activation="linear")(x)
vae_model = Model(vae_input, vae_output)
vae_model.compile(
    optimizer="adam",
    loss="mse"
)
vae_model.fit(
    X_train_common_scaled,
    X_train_common_scaled,
    epochs=10,
    batch_size=512,
    validation_split=0.1,
    verbose=1
)
train_recon = vae_model.predict(X_train_common_scaled, batch_size=512)
train_error = np.mean(np.square(X_train_common_scaled - train_recon), axis=1)
vae_threshold = np.percentile(train_error, 95)
test_recon = vae_model.predict(X_2018_scaled, batch_size=512)
test_error = np.mean(np.square(X_2018_scaled - test_recon), axis=1)
vae_pred = np.where(test_error > vae_threshold, 1, 0)

print("VAE-style Autoencoder Results")
print(confusion_matrix(y_2018, vae_pred))
print(classification_report(
    y_2018,
    vae_pred,
    target_names=["Benign", "Attack"],
    zero_division=0
))

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report

input_dim = X_train_common_scaled.shape[1]
latent_dim = 8

class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        epsilon = tf.random.normal(shape=tf.shape(z_mean))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

class VAE(keras.Model):
    def __init__(self, input_dim, latent_dim):
        super(VAE, self).__init__()

        self.encoder_dense1 = layers.Dense(64, activation="relu")
        self.encoder_dense2 = layers.Dense(32, activation="relu")
        self.z_mean_layer = layers.Dense(latent_dim)
        self.z_log_var_layer = layers.Dense(latent_dim)
        self.sampling = Sampling()

        self.decoder_dense1 = layers.Dense(32, activation="relu")
        self.decoder_dense2 = layers.Dense(64, activation="relu")
        self.decoder_output = layers.Dense(input_dim, activation="linear")

    def encode(self, x):
        x = self.encoder_dense1(x)
        x = self.encoder_dense2(x)
        z_mean = self.z_mean_layer(x)
        z_log_var = self.z_log_var_layer(x)
        z = self.sampling([z_mean, z_log_var])
        return z_mean, z_log_var, z

    def decode(self, z):
        x = self.decoder_dense1(z)
        x = self.decoder_dense2(x)
        return self.decoder_output(x)

    def call(self, inputs):
        z_mean, z_log_var, z = self.encode(inputs)
        reconstructed = self.decode(z)

        reconstruction_loss = tf.reduce_mean(
            tf.reduce_sum(
                tf.square(inputs - reconstructed),
                axis=1
            )
        )
        kl_loss = -0.5 * tf.reduce_mean(
            tf.reduce_sum(
                1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var),
                axis=1
            )
        )
        self.add_loss(reconstruction_loss + 0.001 * kl_loss)
        return reconstructed

vae_model = VAE(
    input_dim=input_dim,
    latent_dim=latent_dim
)
vae_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001)
)
vae_model.fit(
    X_train_common_scaled,
    X_train_common_scaled,
    epochs=10,
    batch_size=512,
    validation_split=0.1,
    verbose=1
)
train_reconstructed = vae_model.predict(
    X_train_common_scaled,
    batch_size=512
)
train_error = np.mean(
    np.square(X_train_common_scaled - train_reconstructed),
    axis=1
)
vae_threshold = np.percentile(
    train_error,
    95
)
test_reconstructed = vae_model.predict(
    X_2018_scaled,
    batch_size=512
)
test_error = np.mean(
    np.square(X_2018_scaled - test_reconstructed),
    axis=1
)
vae_pred = np.where(
    test_error > vae_threshold,
    1,
    0
)
print("Full VAE Results")
print(confusion_matrix(y_2018, vae_pred))
print(classification_report(
    y_2018,
    vae_pred,
    target_names=["Benign", "Attack"],
    zero_division=0
))

In [ ]:
input_dim = X_train_common_scaled.shape[1]
svdd_input = Input(shape=(input_dim,))
x = Dense(64, activation="relu")(svdd_input)
x = Dense(32, activation="relu")(x)
svdd_output = Dense(16, activation="linear")(x)
svdd_model = Model(svdd_input, svdd_output)
initial_embedding = svdd_model.predict(X_train_common_scaled, batch_size=512)
center = np.mean(initial_embedding, axis=0)

def svdd_loss(y_true, y_pred):
    return tf.reduce_mean(tf.reduce_sum(tf.square(y_pred - center), axis=1))
svdd_model.compile(
    optimizer="adam",
    loss=svdd_loss
)
dummy_y = np.zeros((X_train_common_scaled.shape[0], 16))
svdd_model.fit(
    X_train_common_scaled,
    dummy_y,
    epochs=30,
    batch_size=512,
    validation_split=0.1,
    verbose=1
)
train_embedding = svdd_model.predict(X_train_common_scaled, batch_size=512)
train_distance = np.sum(np.square(train_embedding - center), axis=1)
svdd_threshold = np.percentile(train_distance, 95)
test_embedding = svdd_model.predict(X_2018_scaled, batch_size=512)
test_distance = np.sum(np.square(test_embedding - center), axis=1)
svdd_pred = np.where(test_distance > svdd_threshold, 1, 0)

print("Deep SVDD Results")
print(confusion_matrix(y_2018, svdd_pred))
print(classification_report(y_2018, svdd_pred, target_names=["Benign", "Attack"], zero_division=0))

In [ ]:
# ============================================================
# SAVE 2018 EXPERIMENT OUTPUTS
# CICIDS2017 → CSE-CIC-IDS2018
# ============================================================

import os
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# ------------------------------------------------------------
# Save folder
# ------------------------------------------------------------

SAVE_DIR = "/content/drive/MyDrive/Result/2018"
os.makedirs(SAVE_DIR, exist_ok=True)

# ------------------------------------------------------------
# Create final 2018 results table
# ------------------------------------------------------------

results_2018 = []

def add_result(model_name, y_true, y_pred):
    results_2018.append({
        "Dataset": "CICIDS2017_to_CSE_CIC_IDS2018",
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Attack Precision": precision_score(y_true, y_pred, zero_division=0),
        "Attack Recall": recall_score(y_true, y_pred, zero_division=0),
        "Attack F1": f1_score(y_true, y_pred, zero_division=0)
    })

add_result("Isolation Forest", y_2018, y_pred_2018)
add_result("SGD One-Class SVM", y_2018, svm_2018_pred)
add_result("Autoencoder", y_2018, ae_2018_pred)
add_result("VAE", y_2018, vae_pred)
add_result("Deep SVDD-style", y_2018, svdd_pred)

results_2018 = pd.DataFrame(results_2018)

# ------------------------------------------------------------
# Save results table
# ------------------------------------------------------------

results_2018.to_csv(
    f"{SAVE_DIR}/results_2018.csv",
    index=False
)

# ------------------------------------------------------------
# Save common features used for 2018 alignment
# ------------------------------------------------------------

joblib.dump(
    common_features,
    f"{SAVE_DIR}/common_features_2018.joblib"
)

# ------------------------------------------------------------
# Save predictions
# ------------------------------------------------------------

predictions_2018 = pd.DataFrame({
    "True_Label": np.array(y_2018),
    "Isolation_Forest": np.array(y_pred_2018),
    "SGD_OneClass_SVM": np.array(svm_2018_pred),
    "Autoencoder": np.array(ae_2018_pred),
    "VAE": np.array(vae_pred),
    "Deep_SVDD": np.array(svdd_pred)
})

predictions_2018.to_csv(
    f"{SAVE_DIR}/predictions_2018.csv",
    index=False
)

# ------------------------------------------------------------
# Save confusion matrices
# ------------------------------------------------------------

confusion_matrices = {
    "cm_isolation_forest": confusion_matrix(y_2018, y_pred_2018),
    "cm_sgd_ocsvm": confusion_matrix(y_2018, svm_2018_pred),
    "cm_autoencoder": confusion_matrix(y_2018, ae_2018_pred),
    "cm_vae": confusion_matrix(y_2018, vae_pred),
    "cm_deep_svdd": confusion_matrix(y_2018, svdd_pred)
}

for name, matrix in confusion_matrices.items():
    np.save(
        f"{SAVE_DIR}/{name}.npy",
        matrix
    )

# ------------------------------------------------------------
# Save readable confusion matrix table
# ------------------------------------------------------------

cm_rows = []

for model_name, matrix in confusion_matrices.items():
    cm_rows.append({
        "Model": model_name,
        "True_Benign_Pred_Benign": matrix[0, 0],
        "True_Benign_Pred_Attack": matrix[0, 1],
        "True_Attack_Pred_Benign": matrix[1, 0],
        "True_Attack_Pred_Attack": matrix[1, 1]
    })

cm_table_2018 = pd.DataFrame(cm_rows)

cm_table_2018.to_csv(
    f"{SAVE_DIR}/confusion_matrices_2018.csv",
    index=False
)

# ------------------------------------------------------------
# Final check
# ------------------------------------------------------------

print("2018 experiment outputs saved successfully")
print("Saved location:", SAVE_DIR)
print("\nResults:")
display(results_2018)

print("\nConfusion matrices:")
display(cm_table_2018)

print("\nFiles saved:")
print(os.listdir(SAVE_DIR))

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Isolation Forest",
        "SGD One-Class SVM",
        "Autoencoder",
        "VAE Style",
        "Full VAE",
        "Deep SVDD"
    ],
    "Accuracy": [
        0.68,
        0.83,
        0.66,
        0.76,
        0.77,
        0.83
    ],
    "Attack Precision": [
        0.12,
        0.00,
        0.10,
        0.11,
        0.14,
        0.01
    ],

    "Attack Recall": [
        0.13,
        0.00,
        0.13,
        0.06,
        0.07,
        0.00
    ],
    "Attack F1": [
        0.12,
        0.00,
        0.11,
        0.08,
        0.09,
        0.00
    ]
})
print(results)

# NIDS

In [ ]:
import pandas as pd

nids_path = "/content/drive/MyDrive/NIDS/NF-UNSW-NB15-v3.csv"

nids_data = pd.read_csv(
    nids_path,
    low_memory=False
)

print(nids_data.shape)

print("\nColumns:")
print(nids_data.columns.tolist())

print("\nFirst rows:")
display(nids_data.head())

In [ ]:
print(nids_data.shape)
print(nids_data.columns.tolist())

In [ ]:
# ============================================================
# Compare all feature sets
# ============================================================

features_2017 = set(X_train.columns)

features_2018 = set(common_features)

features_nids = set(nids_data.columns)

remove_cols = {
    "Label",
    "Attack",
    "IPV4_SRC_ADDR",
    "IPV4_DST_ADDR",
    "L4_SRC_PORT",
    "L4_DST_PORT",
    "FLOW_START_MILLISECONDS",
    "FLOW_END_MILLISECONDS"
}

features_nids = features_nids - remove_cols

common_3way_features = sorted(
    list(
        features_2017
        .intersection(features_2018)
        .intersection(features_nids)
    )
)

print("2017 features:", len(features_2017))
print("2018 common features:", len(features_2018))
print("NIDS usable features:", len(features_nids))

print("\n3-WAY COMMON FEATURES")
print(len(common_3way_features))

for f in common_3way_features:
    print(f)

In [ ]:
print("=== CICIDS2017 Features ===")
for col in sorted(X_train.columns):
    print(col)

print("\n\n=== NIDS Features ===")
for col in sorted(nids_data.columns):
    print(col)

In [ ]:
nids_subset = nids_data[[
    "FLOW_DURATION_MILLISECONDS",
    "SRC_TO_DST_IAT_MIN",
    "SRC_TO_DST_IAT_MAX",
    "SRC_TO_DST_IAT_AVG",
    "SRC_TO_DST_IAT_STDDEV",
    "DST_TO_SRC_IAT_MIN",
    "DST_TO_SRC_IAT_MAX",
    "DST_TO_SRC_IAT_AVG",
    "DST_TO_SRC_IAT_STDDEV",
    "MIN_IP_PKT_LEN",
    "MAX_IP_PKT_LEN",
    "IN_BYTES",
    "OUT_BYTES",
    "IN_PKTS",
    "OUT_PKTS"
]]

print(nids_subset.describe().T)

In [ ]:
cicids_subset = X_train[[
    "Flow Duration",
    "Fwd IAT Min",
    "Fwd IAT Max",
    "Fwd IAT Mean",
    "Fwd IAT Std",
    "Bwd IAT Min",
    "Bwd IAT Max",
    "Bwd IAT Mean",
    "Bwd IAT Std",
    "Min Packet Length",
    "Max Packet Length",
    "Total Length of Fwd Packets",
    "Total Length of Bwd Packets",
    "Total Fwd Packets",
    "Total Backward Packets"
]]

print(cicids_subset.describe().T)

In [ ]:
# ============================================================
# EXPERIMENT: CICIDS2017 → NF-UNSW-NB15-v3
# Manual Semantic Feature Alignment
# ============================================================

import os
import numpy as np
import pandas as pd
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import SGDOneClassSVM
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# ------------------------------------------------------------
# Save folder
# ------------------------------------------------------------

SAVE_DIR = "/content/drive/MyDrive/Result/NIDS"
os.makedirs(SAVE_DIR, exist_ok=True)

# ------------------------------------------------------------
# Manual feature mapping
# CICIDS2017 feature -> NIDS feature
# ------------------------------------------------------------

feature_mapping = {
    "Flow Duration": "FLOW_DURATION_MILLISECONDS",

    "Fwd IAT Min": "SRC_TO_DST_IAT_MIN",
    "Fwd IAT Max": "SRC_TO_DST_IAT_MAX",
    "Fwd IAT Mean": "SRC_TO_DST_IAT_AVG",
    "Fwd IAT Std": "SRC_TO_DST_IAT_STDDEV",

    "Bwd IAT Min": "DST_TO_SRC_IAT_MIN",
    "Bwd IAT Max": "DST_TO_SRC_IAT_MAX",
    "Bwd IAT Mean": "DST_TO_SRC_IAT_AVG",
    "Bwd IAT Std": "DST_TO_SRC_IAT_STDDEV",

    "Min Packet Length": "MIN_IP_PKT_LEN",
    "Max Packet Length": "MAX_IP_PKT_LEN"
}

# ------------------------------------------------------------
# Check all mapped features exist
# ------------------------------------------------------------

missing_2017 = [c for c in feature_mapping.keys() if c not in X_train.columns]
missing_nids = [c for c in feature_mapping.values() if c not in nids_data.columns]

print("Missing CICIDS2017 features:", missing_2017)
print("Missing NIDS features:", missing_nids)

if len(missing_2017) > 0 or len(missing_nids) > 0:
    raise ValueError("Some mapped features are missing. Check feature names.")

# ------------------------------------------------------------
# Prepare CICIDS2017 benign training data
# ------------------------------------------------------------

X_train_manual = X_train[list(feature_mapping.keys())].copy()

for col in X_train_manual.columns:
    X_train_manual[col] = pd.to_numeric(X_train_manual[col], errors="coerce")

X_train_manual.replace([np.inf, -np.inf], np.nan, inplace=True)
X_train_manual.fillna(X_train_manual.median(), inplace=True)

print("CICIDS2017 manual training shape:", X_train_manual.shape)

# ------------------------------------------------------------
# Prepare NIDS test data
# ------------------------------------------------------------

X_nids_manual = nids_data[list(feature_mapping.values())].copy()

for col in X_nids_manual.columns:
    X_nids_manual[col] = pd.to_numeric(X_nids_manual[col], errors="coerce")

X_nids_manual.replace([np.inf, -np.inf], np.nan, inplace=True)

# Rename NIDS columns to CICIDS2017 feature names
X_nids_manual.columns = list(feature_mapping.keys())

# Fill NIDS missing values using CICIDS2017 training medians
X_nids_manual.fillna(X_train_manual.median(), inplace=True)

print("NIDS manual test shape:", X_nids_manual.shape)
print("Missing values in NIDS:", X_nids_manual.isnull().sum().sum())

# ------------------------------------------------------------
# Prepare NIDS labels
# ------------------------------------------------------------

print("Original NIDS label values:")
print(nids_data["Label"].value_counts())

y_nids = nids_data["Label"].apply(
    lambda x: 0 if str(x).strip().lower() in ["benign", "normal", "0"] else 1
)

print("Binary NIDS labels:")
print(y_nids.value_counts())

# ------------------------------------------------------------
# Scale using CICIDS2017 benign training only
# ------------------------------------------------------------

manual_scaler = StandardScaler()

X_train_manual_scaled = manual_scaler.fit_transform(X_train_manual)
X_nids_manual_scaled = manual_scaler.transform(X_nids_manual)

print("Scaled CICIDS2017 shape:", X_train_manual_scaled.shape)
print("Scaled NIDS shape:", X_nids_manual_scaled.shape)

# ------------------------------------------------------------
# Model 1: Isolation Forest
# ------------------------------------------------------------

iso_nids = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42,
    n_jobs=-1
)

iso_nids.fit(X_train_manual_scaled)

iso_raw = iso_nids.predict(X_nids_manual_scaled)
iso_nids_pred = np.where(iso_raw == 1, 0, 1)

print("\nIsolation Forest: CICIDS2017 → NF-UNSW-NB15-v3")
print(confusion_matrix(y_nids, iso_nids_pred))
print(classification_report(
    y_nids,
    iso_nids_pred,
    target_names=["Benign", "Attack"],
    zero_division=0
))

# ------------------------------------------------------------
# Model 2: SGD One-Class SVM
# ------------------------------------------------------------

svm_nids = SGDOneClassSVM(
    nu=0.05,
    random_state=42
)

svm_nids.fit(X_train_manual_scaled)

svm_raw = svm_nids.predict(X_nids_manual_scaled)
svm_nids_pred = np.where(svm_raw == 1, 0, 1)

print("\nSGD One-Class SVM: CICIDS2017 → NF-UNSW-NB15-v3")
print(confusion_matrix(y_nids, svm_nids_pred))
print(classification_report(
    y_nids,
    svm_nids_pred,
    target_names=["Benign", "Attack"],
    zero_division=0
))

# ------------------------------------------------------------
# Results table
# ------------------------------------------------------------

results_nids = []

def add_nids_result(model_name, y_true, y_pred):
    results_nids.append({
        "Dataset": "CICIDS2017_to_NF_UNSW_NB15_v3_manual_alignment",
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Attack Precision": precision_score(y_true, y_pred, zero_division=0),
        "Attack Recall": recall_score(y_true, y_pred, zero_division=0),
        "Attack F1": f1_score(y_true, y_pred, zero_division=0)
    })

add_nids_result("Isolation Forest", y_nids, iso_nids_pred)
add_nids_result("SGD One-Class SVM", y_nids, svm_nids_pred)

results_nids = pd.DataFrame(results_nids)

display(results_nids)

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

results_nids.to_csv(
    f"{SAVE_DIR}/results_nids_manual_alignment.csv",
    index=False
)

pd.DataFrame({
    "CICIDS2017_Feature": list(feature_mapping.keys()),
    "NIDS_Feature": list(feature_mapping.values())
}).to_csv(
    f"{SAVE_DIR}/manual_feature_mapping.csv",
    index=False
)

pd.DataFrame({
    "True_Label": np.array(y_nids),
    "Isolation_Forest": np.array(iso_nids_pred),
    "SGD_OneClass_SVM": np.array(svm_nids_pred)
}).to_csv(
    f"{SAVE_DIR}/predictions_nids_manual_alignment.csv",
    index=False
)

joblib.dump(
    feature_mapping,
    f"{SAVE_DIR}/manual_feature_mapping.joblib"
)

joblib.dump(
    manual_scaler,
    f"{SAVE_DIR}/manual_alignment_scaler.joblib"
)

joblib.dump(
    iso_nids,
    f"{SAVE_DIR}/isolation_forest_nids_manual.joblib"
)

joblib.dump(
    svm_nids,
    f"{SAVE_DIR}/sgd_ocsvm_nids_manual.joblib"
)

print("\nNIDS manual alignment experiment saved successfully")
print("Saved location:", SAVE_DIR)
print(os.listdir(SAVE_DIR))

In [ ]:
# ============================================================
# SAVE NIDS MANUAL ALIGNMENT RESULTS
# ============================================================

import os
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

SAVE_DIR = "/content/drive/MyDrive/Result/NIDS"
os.makedirs(SAVE_DIR, exist_ok=True)

results_nids.to_csv(
    f"{SAVE_DIR}/results_nids_manual_alignment.csv",
    index=False
)

pd.DataFrame({
    "CICIDS2017_Feature": list(feature_mapping.keys()),
    "NIDS_Feature": list(feature_mapping.values())
}).to_csv(
    f"{SAVE_DIR}/manual_feature_mapping.csv",
    index=False
)

pd.DataFrame({
    "True_Label": np.array(y_nids),
    "Isolation_Forest": np.array(iso_nids_pred),
    "SGD_OneClass_SVM": np.array(svm_nids_pred)
}).to_csv(
    f"{SAVE_DIR}/predictions_nids_manual_alignment.csv",
    index=False
)

np.save(
    f"{SAVE_DIR}/cm_isolation_forest_nids.npy",
    confusion_matrix(y_nids, iso_nids_pred)
)

np.save(
    f"{SAVE_DIR}/cm_sgd_ocsvm_nids.npy",
    confusion_matrix(y_nids, svm_nids_pred)
)

joblib.dump(feature_mapping, f"{SAVE_DIR}/manual_feature_mapping.joblib")
joblib.dump(manual_scaler, f"{SAVE_DIR}/manual_alignment_scaler.joblib")
joblib.dump(iso_nids, f"{SAVE_DIR}/isolation_forest_nids_manual.joblib")
joblib.dump(svm_nids, f"{SAVE_DIR}/sgd_ocsvm_nids_manual.joblib")

print("NIDS results saved successfully")
print(os.listdir(SAVE_DIR))

In [ ]:
combined_results = pd.concat(
    [
        results_2017,
        results_2018,
        results_nids
    ],
    ignore_index=True
)

combined_results = combined_results[
    [
        "Dataset",
        "Model",
        "Accuracy",
        "Attack Precision",
        "Attack Recall",
        "Attack F1"
    ]
]

display(combined_results)

combined_results.to_csv(
    "/content/drive/MyDrive/Result/final_combined_results.csv",
    index=False
)